# Step 1: Data Preprocessing
In this notebook, we prepare the raw medical dataset for machine learning models. The detailed preprocessing pipeline includes:
* **Duplicate Removal:** Dropping duplicate records to prevent data leakage.
* **One-Hot Encoding:** Converting categorical variables (`gender`, `smoking_history`) into binary dummy columns.
* **Outlier Capping (IQR):** Handling extreme medical anomalies in BMI and glucose levels using the Interquartile Range method to prevent model distortion.
* **Feature Engineering:** Creating a new `health_risk_score` metric combining existing conditions like hypertension and heart disease.
* **Class Balancing (SMOTE):** Generating synthetic samples for the minority (diabetic) class to tackle the 91.5% imbalance.
* **Feature Scaling:** Applying `StandardScaler` to ensure distance-based bounds and estimators perform optimally.


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

# 1. Load dataset
df = pd.read_csv('diabetes_prediction_dataset.csv')
print(f"Original dataset size: {df.shape}")

# Handle Duplicates
df = df.drop_duplicates()

# 2. Advanced Preprocessing: Outlier Handling with IQR (on 'bmi', 'HbA1c_level', etc.) 
# We'll gently cap outliers using the 1.5*IQR rule rather than dropping them, to preserve valuable medical edge-cases.
numeric_cols = ['bmi', 'HbA1c_level', 'blood_glucose_level']
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])

# 3. Feature Engineering
# Create a new feature combining hypertension and heart disease risk
df['health_risk_score'] = df['hypertension'] + df['heart_disease']

# 4. One-Hot Encoding for categorical variables instead of Label Encoding
# This prevents numerical hierarchical bias for 'gender' and 'smoking_history'
df = pd.get_dummies(df, columns=['gender', 'smoking_history'], drop_first=True)

# Print the top 10 rows and current size right after all preprocessing
print("\n--- After Preprocessing & Feature Engineering ---")
print(f"Dataset Size (rows, columns): {df.shape}")
print("\nTop 10 rows:")
display(df.head(10))  # display automatically formats it in jupyter beautifully

# 5. Features & Target Setup
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# 6. Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to dataframe to keep column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

# 8. Handle Imbalance with SMOTE on Training Data ONLY
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Save processed data via joblib
joblib.dump((X_train_resampled, y_train_resampled, X_test_scaled, y_test), 'processed_data.pkl')
joblib.dump(X.columns.tolist(), 'feature_names.pkl') # Save feature names for potential future use

print("\nData Preprocessing Complete!")
print("Resampled Target Distribution:")
print(pd.Series(y_train_resampled).value_counts())

Original dataset size: (100000, 9)

--- After Preprocessing & Feature Engineering ---
Dataset Size (rows, columns): (96146, 15)

Top 10 rows:


,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes,health_risk_score,gender_Male,gender_Other,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current
0,80.0,0,1,25.19,6.6,140.0,0,1,False,False,False,False,False,True,False
1,54.0,0,0,27.32,6.6,80.0,0,0,False,False,False,False,False,False,False
2,28.0,0,0,27.32,5.7,158.0,0,0,True,False,False,False,False,True,False
3,36.0,0,0,23.45,5.0,155.0,0,0,False,False,True,False,False,False,False
4,76.0,1,1,20.14,4.8,155.0,0,2,True,False,True,False,False,False,False
5,20.0,0,0,27.32,6.6,85.0,0,0,False,False,False,False,False,True,False
6,44.0,0,0,19.31,6.5,200.0,1,0,False,False,False,False,False,True,False
7,79.0,0,0,23.86,5.7,85.0,0,0,False,False,False,False,False,False,False
8,42.0,0,0,33.64,4.8,145.0,0,0,True,False,False,False,False,True,False
9,32.0,0,0,27.32,5.0,100.0,0,0,False,False,False,False,False,True,False



Data Preprocessing Complete!
Resampled Target Distribution:
diabetes
0    70130
1    70130
Name: count, dtype: int64
